# Week 4, day 2 (afternoon) — dbt on Snowflake: from raw to data marts

The WeCloudData **Create a dbt Project** lab and the **dbt Fundamentals**
lecture, run entirely inside Snowflake.

You create the warehouse, database, schemas, raw tables, file format and stages;
load the data; then deploy a complete dbt project — sources, staging, snapshots,
a **Type 6** slowly-changing dimension, a **star schema** on surrogate keys,
seeds that answer real business requirements, custom generic tests, unit tests
and a **MetricFlow semantic layer** — and run it as a native `DBT PROJECT`
object.

## What is where

| | |
|---|---|
| **This notebook** | every Snowflake statement: create objects, load, deploy, run dbt, inspect, schedule, tear down. All SQL. |
| **`dbt-project/demo/`** | the dbt project as real files, in the repo. You upload this folder to a stage. Read its `README.md` first. |

**There are no credentials here.** A Snowflake notebook is already
authenticated, and dbt runs as the role that executes the project.

## Before you start

1. A role that can create a database, schemas, stages, tasks and a `DBT PROJECT`,
   with a warehouse attached to this notebook.
2. The two CSVs to hand — `products.csv` (1,214 rows) and `sales.csv`
   (100,000 rows). Question 7 uploads them.
3. The `dbt-project/demo/` folder to hand. Question 9 uploads it.

## The layers

```
RAW  ──▶  STG  ──▶  EDW  ──▶  MARTS
 │         │         │          │
 │         │         │          ├─ rpt_*   hand-written SQL models
 │         │         │          └─ mart_*  exported from MetricFlow saved queries
 │         │         └─ dim_product_t6 (Type 6) / dim_store / dim_date / fct_sales
 │         └─ stg_product_incr / stg_sales (views) + the snapshots
 └─ product / sales  (the two CSVs)      + seeds: store_master, category_targets
```

New to dbt? The day's `README.md` opens with a concepts primer — model, source,
ref, seed, snapshot, materialization, macro, test, and the three ways to
configure a model. Read that first.

Work the questions in order: each depends on what the earlier ones made.


Run this first, every session.

In [ ]:
-- Run this first. It sets the session context every later cell assumes.
-- There are NO credentials in this notebook: a Snowflake notebook is already
-- authenticated, and dbt runs as the role executing the project.
USE ROLE SYSADMIN;              -- or whichever role owns the lab objects
USE WAREHOUSE COMPUTE_WH;       -- change if your warehouse is named differently
USE DATABASE DEMO_DB;

SELECT CURRENT_ROLE()      AS my_role,
       CURRENT_WAREHOUSE() AS my_warehouse,
       CURRENT_DATABASE()  AS my_database,
       CURRENT_VERSION()   AS snowflake_version;

## PART A — create the objects

Everything the lab needs, made explicitly. dbt creates objects *inside* these; it
never creates the database, the schemas or the raw tables.

### Question 1

Create the **warehouse**, and check your role can do what follows.
Skip the `CREATE` if you already have one.

In [ ]:
-- Your Code Here

### Question 2

**Optional, ACCOUNTADMIN only.** A dedicated least-privilege role for the
lab. Skip it if you are running as `SYSADMIN`.

In [ ]:
-- Your Code Here

### Question 3

Create the database and the five schemas — one per layer.

In [ ]:
-- Your Code Here

### Question 4

Create the two raw tables, matching the CSV headers exactly.

In [ ]:
-- Your Code Here

### Question 5

Create a named **file format**, so the parsing rules are one object every
`COPY` reuses.

In [ ]:
-- Your Code Here

### Question 6

Create the two **stages**: one for the CSVs, one for the dbt project.

In [ ]:
-- Your Code Here

### Question 7

**Upload the two CSVs** to `RAW.LOAD_STAGE`, then load them.

In Snowsight: **Data → Databases → DEMO_DB → RAW → Stages → LOAD_STAGE →
+ Files**. Then run this.

In [ ]:
-- Your Code Here

### Question 8

Verify the load — row counts, and the date range that proves the format
was applied.

In [ ]:
-- Your Code Here

## PART B — deploy and run the dbt project

The project is a folder of files in this repo at
`week4_day2_afternoon/dbt-project/demo/`. You upload it to the stage, turn it
into a `DBT PROJECT` object, and run dbt as SQL.

Read that folder's `README.md` before uploading — it lists exactly what to
include, and explains why `packages.yml` is empty.

### Question 9

**Upload the project folder** to `@DEMO_DB.RAW.DBT_PROJECT_STAGE/demo/`,
preserving its directory structure, then confirm what landed.

`dbt_project.yml` must sit at the root of that prefix. Exclude `target/`,
`logs/` and `profiles.example.yml`.

In [ ]:
-- Your Code Here

### Question 10

Create the **dbt project object** from the stage.

In [ ]:
-- Your Code Here

### Question 11

Build everything: seeds, snapshots, models and tests, in dependency order.

In [ ]:
-- Your Code Here

### Question 12

Run one layer at a time, one model, and a full refresh — the same
selectors as the CLI.

In [ ]:
-- Your Code Here

### Question 13

Materialize the semantic layer's saved queries — the extra marts.

In [ ]:
-- Your Code Here

## PART C — inspect what dbt built

The interesting part. Everything below is a plain query against objects dbt
created.

### Question 14

Look at staging and the two snapshots.

In [ ]:
-- Your Code Here

### Question 15

Look at the **Type 6** dimension — all three views on one row.

In [ ]:
-- Your Code Here

### Question 16

Prove the **surrogate key** is what makes the dimension addressable.

In [ ]:
-- Your Code Here

### Question 17

Query the **star**: the fact joined to all three dimensions.

In [ ]:
-- Your Code Here

### Question 18

Look at the marts, including the one the **seed** exists for.

In [ ]:
-- Your Code Here

### Question 19

**Make history happen.** Change a product, re-snapshot, rebuild, and watch
the Type 6 columns diverge.

In [ ]:
-- Your Code Here

## PART D — lineage across the layers

dbt knows the whole graph because every model declares its inputs with `ref()`
and `source()`. Both directions are answerable — and the second is the question
to ask *before* editing anything.

### Question 20

Trace **upstream**: everything that feeds the regional sales report.

In [ ]:
-- Your Code Here

### Question 21

Trace **downstream**: the blast radius if `stg_sales` changed. Then what a
dashboard depends on.

In [ ]:
-- Your Code Here

### Question 22

Confirm the layers landed where the project's routing said they would.

In [ ]:
-- Your Code Here

## PART E — what dbt actually wrote, scheduling, and cleanup

### Question 23

Ask Snowflake for the **DDL** of what dbt built.

In [ ]:
-- Your Code Here

### Question 24

Schedule it. Inside Snowflake the scheduler is already there — a `TASK`,
no cron and no external orchestrator.

In [ ]:
-- Your Code Here

### Question 25

Check the task history — how you find out a scheduled run failed.

In [ ]:
-- Your Code Here

### Question 26

**Tear it down** — so you can re-run the class from scratch, and so nothing
keeps billing.

In [ ]:
-- Your Code Here